<class 'ImportError'>: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [5]:
import os
print(os.listdir('.'))

['BTT.ipynb', 'BreakThroughTech_training_data_1.xlsx', 'data', 'notebooks', 'README.md']


In [7]:
import micropip
await micropip.install('openpyxl')

In [8]:
import pandas as pd
import numpy as np

df = pd.read_excel('BreakThroughTech_training_data_1.xlsx')
df.head()

,brand,sub_product_name,country,pillar,media_type,profession,specialty,campaign_start_date,campaign_end_date,unique_reach,...,post_avg_NRx,pre_NRx_avg,npi_specialty,region,NRx_lift,NRx_lift_pct,cpe,total_spend,actual_spend_per_engagement,spend_per_NRx_lift
0,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,8,undefined,2024-07-31 10:48:34.000000 UTC,2099-12-31 23:59:59.999999 UTC,9270,...,72.8343,47.4714,Neurology,Southeast,146.77,15.5000,39.0,134355308.0,30198.99,915440.73
1,ABBVIE,BRAND ALERT,usa,search,EDITORIAL_VIDEOS,8,undefined,2024-08-19 17:16:44.000000 UTC,2099-12-31 23:59:59.999999 UTC,1959,...,54.1279,62.6833,Cardiology,Southwest,112.05,12.2398,39.0,10667850.0,9055.90,95202.91
2,ABBVIE,BRAND ALERT,usa,search,SPONSORED_VIDEOS,psychologist,urology,2023-11-16 15:54:10.000000 UTC,2099-12-31 23:59:59.999999 UTC,2860,...,42.1019,66.2213,Cardiology,Mid-Atlantic,78.27,10.8516,39.0,59103000.0,34705.23,755161.42
3,ABBVIE,BRAND ALERT,zaf,search,SPONSORED_VIDEOS,8,undefined,2024-11-22 10:27:31.000000 UTC,2099-12-31 23:59:59.999999 UTC,7390,...,49.5424,55.8293,Neurology,Northeast,121.87,11.9201,39.0,124487739.0,112455.05,1021479.17
4,ABBVIE,EMAIL ALERT,NaN,NaN,NaN,NaN,NaN,2023-11-16 17:38:18.000000 UTC,2099-12-31 23:59:59.999999 UTC,77570,...,46.8907,39.3346,Endocrinology,Southwest,221.51,35.1116,3.0,892409925.0,7691.40,4028833.69


In [9]:
cols = ['NRx_lift_pct', 'spend_per_NRx_lift']

# 1. Missing values
missing = df[cols].isnull().sum()
missing_pct = df[cols].isnull().mean() * 100
print(missing, missing_pct)

# 2. Dtypes and basic stats
print(df[cols].dtypes)
print(df[cols].describe())

# 3. NRx_lift_pct — check for invalid percentage ranges
invalid_pct = df[(df['NRx_lift_pct'] < -100) | (df['NRx_lift_pct'] > 1000)]
print(f"NRx_lift_pct out-of-range rows: {len(invalid_pct)}")

# 4. spend_per_NRx_lift — check for inf/negative/zero issues
inf_rows = df[np.isinf(df['spend_per_NRx_lift'])]
negative_rows = df[df['spend_per_NRx_lift'] < 0]
zero_rows = df[df['spend_per_NRx_lift'] == 0]
print(f"Inf: {len(inf_rows)}, Negative: {len(negative_rows)}, Zero: {len(zero_rows)}")

# 5. Outliers via IQR
for c in cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = df[(df[c] < lower) | (df[c] > upper)]
    print(f"{c}: {len(outliers)} outliers (bounds: {lower:.2f} to {upper:.2f})")

# 6. Duplicate check
print(f"Full duplicate rows: {df.duplicated().sum()}")

NRx_lift_pct          0
spend_per_NRx_lift    0
dtype: int64 NRx_lift_pct          0.0
spend_per_NRx_lift    0.0
dtype: float64
NRx_lift_pct          float64
spend_per_NRx_lift    float64
dtype: object
       NRx_lift_pct  spend_per_NRx_lift
count    417.000000        4.170000e+02
mean      20.604140        5.555055e+06
std        9.392379        1.438881e+07
min        2.708800        0.000000e+00
25%       13.264700        2.322284e+05
50%       19.109900        9.154407e+05
75%       26.264600        4.028834e+06
max       52.650100        1.288480e+08
NRx_lift_pct out-of-range rows: 0
Inf: 0, Negative: 0, Zero: 2
NRx_lift_pct: 8 outliers (bounds: -6.24 to 45.76)
spend_per_NRx_lift: 49 outliers (bounds: -5462679.54 to 9723741.62)
Full duplicate rows: 0
